# 🧠 Tool Error Handling & Retries

This notebook explains **how GenAI systems must handle tool failures**
without letting LLMs improvise, panic, or corrupt state.

You will learn:
- Why tool failures are normal in production
- Why LLMs must never manage errors
- Deterministic retry strategies
- When to retry, fail, fallback, or refuse
- How to make failures observable and safe

📌 Core rule:
> Errors are system events, not language events.


## 1. Tool Failures Are Inevitable

In production, tools fail due to:
- network issues
- timeouts
- rate limits
- invalid inputs
- downstream outages

A system that assumes success
will fail catastrophically.


## 2. Dangerous Anti-Pattern

❌ Letting the LLM:
- interpret error messages
- decide whether to retry
- “fix” arguments
- guess alternative actions

This turns errors into hallucinations.


## 3. Responsibility Split

LLM:
- explains failures to the user
- suggests next steps in natural language

System:
- detects errors
- classifies error type
- applies retry logic
- enforces limits


## 4. Error Taxonomy

Tool errors fall into categories:

1. Transient errors (retryable)
2. Deterministic errors (not retryable)
3. Permission / policy errors
4. Partial execution errors
5. Unknown / unexpected errors

Each requires a different response.


## 5. Transient Errors

Examples:
- network timeout
- temporary service outage
- rate limit exceeded

Correct response:
- retry deterministically
- apply backoff
- cap retries


## 6. Deterministic Errors

Examples:
- invalid input
- schema violation
- business rule failure

Retrying will:
- waste resources
- repeat failure

Correct response:
> Fail fast and explain.


## 7. Retry Policy Design

A safe retry policy defines:
- which errors are retryable
- max retry count
- backoff strategy
- timeout per attempt

Retries must be:
> predictable and bounded


In [ ]:
RETRYABLE_ERRORS = {"timeout", "rate_limit"}
MAX_RETRIES = 3

def should_retry(error_type, attempt):
    return error_type in RETRYABLE_ERRORS and attempt < MAX_RETRIES


In [ ]:
def execute_with_retries(tool_fn, args):
    attempt = 0
    while True:
        try:
            return tool_fn(args)
        except Exception as e:
            error_type = getattr(e, "type", "unknown")
            attempt += 1
            if should_retry(error_type, attempt):
                continue
            else:
                raise


## 10. Backoff Strategy

Without backoff:
- retries amplify outages
- systems cascade-fail

Common backoff:
- exponential
- jittered

Never retry immediately in a loop.


## 11. Partial Execution

Partial execution means:
- some side effects occurred
- but the operation failed

This is worse than total failure.

Mitigation:
- idempotency keys
- transactional boundaries
- compensating actions


## 12. Idempotency

Idempotent tools ensure:
- retries do not duplicate effects
- crashes do not corrupt state

Retries without idempotency
are a data corruption vector.


## 13. Decision Matrix

Retry:
- transient failure
- same action, same args

Fallback:
- secondary data source
- lower-quality but safe result

Refuse:
- policy violation
- missing permissions
- unsafe request


## 14. LLM Visibility

The LLM should receive:
- high-level error category
- user-safe explanation

The LLM should NOT see:
- stack traces
- raw exceptions
- internal system state


## 15. Observability

Log every failure with:
- tool name
- error category
- retry count
- final outcome
- timestamps

If errors are invisible,
they will repeat.


## Final Mental Lock

LLMs must never:
- interpret errors
- fix failures
- retry tools

Failures are handled by systems.
Explanations are handled by language.


## Self-Check

You understand this notebook if you can explain:

- Why retries must be deterministic
- Why some errors should never be retried
- Why partial execution is dangerous
- Why LLMs must not handle failures


GenAI systems don’t fail because tools break.

They fail because
failure handling was improvised.

Deterministic error handling
is what turns GenAI into real software.
